 -- ============================================================
# - **00_setup_catalog_schema.sql**
- **Enterprise Databricks Lakehouse Platform (Market + Macro)**
> Schemas (Domains x Layers)


In [0]:
CREATE SCHEMA IF NOT EXISTS enterprise.bronze_market;
CREATE SCHEMA IF NOT EXISTS enterprise.silver_market;
CREATE SCHEMA IF NOT EXISTS enterprise.gold_market;

CREATE SCHEMA IF NOT EXISTS enterprise.bronze_ref;
CREATE SCHEMA IF NOT EXISTS enterprise.silver_ref;
CREATE SCHEMA IF NOT EXISTS enterprise.gold_ref;

CREATE SCHEMA IF NOT EXISTS enterprise.gold_observability;


**Tables - Bronze (Raw)**


In [0]:
CREATE TABLE IF NOT EXISTS enterprise.bronze_market.crypto_ohlc_raw (
  source        STRING COMMENT 'Data provider, e.g., binance/coinbase',
  symbol        STRING COMMENT 'Trading pair symbol, e.g., BTCUSDT',
  interval      STRING COMMENT 'Bar interval, e.g., 1m',
  event_time    TIMESTAMP COMMENT 'Event time from source (if available)',
  raw_json      STRING COMMENT 'Raw JSON payload',
  run_id        STRING COMMENT 'Pipeline run id for replay/audit',
  ingestion_ts  TIMESTAMP COMMENT 'Ingestion timestamp (UTC)'
)
USING DELTA
COMMENT 'Bronze raw OHLC payloads for crypto market data';

CREATE TABLE IF NOT EXISTS enterprise.bronze_market.crypto_trades_raw (
  source        STRING COMMENT 'Data provider',
  symbol        STRING COMMENT 'Trading pair symbol',
  event_time    TIMESTAMP COMMENT 'Trade event time',
  raw_json      STRING COMMENT 'Raw JSON payload',
  run_id        STRING COMMENT 'Pipeline run id for replay/audit',
  ingestion_ts  TIMESTAMP COMMENT 'Ingestion timestamp (UTC)'
)
USING DELTA
COMMENT 'Bronze raw trades payloads (optional, adds depth for platform demo)';

CREATE TABLE IF NOT EXISTS enterprise.bronze_ref.ecb_fx_raw (
  base_ccy      STRING COMMENT 'Base currency (ECB typically EUR)',
  raw_json      STRING COMMENT 'Raw JSON/XML payload (stored as string)',
  run_id        STRING COMMENT 'Pipeline run id for replay/audit',
  ingestion_ts  TIMESTAMP COMMENT 'Ingestion timestamp (UTC)'
)
USING DELTA
COMMENT 'Bronze raw ECB FX reference rates payloads';

CREATE TABLE IF NOT EXISTS enterprise.bronze_ref.fred_series_raw (
  series_id     STRING COMMENT 'FRED series id, e.g., FEDFUNDS, CPIAUCSL',
  raw_json      STRING COMMENT 'Raw JSON payload (stored as string)',
  run_id        STRING COMMENT 'Pipeline run id for replay/audit',
  ingestion_ts  TIMESTAMP COMMENT 'Ingestion timestamp (UTC)'
)
USING DELTA
COMMENT 'Bronze raw FRED macro series payloads';



**Tables - Silver (Curated / Standardized)**

In [0]:
CREATE TABLE IF NOT EXISTS enterprise.silver_market.crypto_ohlc_1m (
  source        STRING COMMENT 'Data provider',
  symbol        STRING COMMENT 'Trading pair symbol',
  bar_start_ts  TIMESTAMP COMMENT 'Bar start timestamp (UTC)',
  bar_end_ts    TIMESTAMP COMMENT 'Bar end timestamp (UTC)',
  open          DOUBLE  COMMENT 'Open price',
  high          DOUBLE  COMMENT 'High price',
  low           DOUBLE  COMMENT 'Low price',
  close         DOUBLE  COMMENT 'Close price',
  volume        DOUBLE  COMMENT 'Volume',
  ingestion_ts  TIMESTAMP COMMENT 'Ingestion timestamp (UTC)',
  p_date        DATE COMMENT 'Partition date derived from bar_start_ts'
)
USING DELTA
PARTITIONED BY (p_date)
COMMENT 'Silver standardized OHLC 1-minute bars (deduped, validated)';

CREATE TABLE IF NOT EXISTS enterprise.silver_market.crypto_trades (
  source         STRING  COMMENT 'Data provider',
  symbol         STRING  COMMENT 'Trading pair symbol',
  trade_id       STRING  COMMENT 'Unique trade id from source',
  trade_ts       TIMESTAMP COMMENT 'Trade timestamp (UTC)',
  price          DOUBLE  COMMENT 'Trade price',
  qty            DOUBLE  COMMENT 'Trade quantity',
  is_buyer_maker BOOLEAN COMMENT 'Source flag (if available)',
  ingestion_ts   TIMESTAMP COMMENT 'Ingestion timestamp (UTC)',
  p_date         DATE COMMENT 'Partition date derived from trade_ts'
)
USING DELTA
PARTITIONED BY (p_date)
COMMENT 'Silver standardized trades (deduped)';

CREATE TABLE IF NOT EXISTS enterprise.silver_ref.ecb_fx_daily (
  rate_date     DATE   COMMENT 'FX rate date',
  base_ccy      STRING COMMENT 'Base currency (ECB typically EUR)',
  quote_ccy     STRING COMMENT 'Quote currency, e.g., USD',
  fx_rate       DOUBLE COMMENT 'FX rate (base->quote)',
  ingestion_ts  TIMESTAMP COMMENT 'Ingestion timestamp (UTC)'
)
USING DELTA
PARTITIONED BY (rate_date)
COMMENT 'Silver daily ECB FX reference rates';

CREATE TABLE IF NOT EXISTS enterprise.silver_ref.fred_series_daily (
  series_id     STRING COMMENT 'FRED series id',
  obs_date      DATE   COMMENT 'Observation date',
  value         DOUBLE COMMENT 'Observation value',
  ingestion_ts  TIMESTAMP COMMENT 'Ingestion timestamp (UTC)'
)
USING DELTA
PARTITIONED BY (obs_date)
COMMENT 'Silver daily FRED macro time series';



**Tables - Gold (Marts / Analytics-ready)**

In [0]:
CREATE TABLE IF NOT EXISTS enterprise.gold_market.ohlc_1m (
  source        STRING,
  symbol        STRING,
  bar_start_ts  TIMESTAMP,
  bar_end_ts    TIMESTAMP,
  open          DOUBLE,
  high          DOUBLE,
  low           DOUBLE,
  close         DOUBLE,
  volume        DOUBLE,
  p_date        DATE,
  mart_ts       TIMESTAMP
)
USING DELTA
PARTITIONED BY (p_date)
COMMENT 'Gold mart: analytics-ready OHLC 1m (from silver)';

CREATE TABLE IF NOT EXISTS enterprise.gold_market.volatility_daily (
  source        STRING,
  symbol        STRING,
  trade_date    DATE,
  bars          BIGINT,
  volatility    DOUBLE,
  daily_volume  DOUBLE,
  day_close     DOUBLE,
  vol_mean_30   DOUBLE,
  vol_std_30    DOUBLE,
  is_anomaly    BOOLEAN,
  p_date        DATE,
  mart_ts       TIMESTAMP
)
USING DELTA
PARTITIONED BY (p_date)
COMMENT 'Gold mart: daily volatility metrics + anomaly flag';

CREATE TABLE IF NOT EXISTS enterprise.gold_market.market_macro_daily (
  source         STRING,
  symbol         STRING,
  trade_date     DATE,
  close_px       DOUBLE,
  daily_volume   DOUBLE,
  eurusd_rate    DOUBLE,
  fedfunds       DOUBLE,
  mart_ts        TIMESTAMP
)
USING DELTA
PARTITIONED BY (trade_date)
COMMENT 'Gold mart: market + macro joined daily dataset';



**Observability (Platform-level metrics)**

In [0]:
CREATE TABLE IF NOT EXISTS enterprise.gold_observability.pipeline_metrics_daily (
  metric_date        DATE COMMENT 'Date of metrics',
  pipeline           STRING COMMENT 'Pipeline stage/job name',
  table_name         STRING COMMENT 'Measured table',
  row_count          BIGINT COMMENT 'Row count',
  max_event_ts       TIMESTAMP COMMENT 'Latest event timestamp in data',
  freshness_minutes  DOUBLE COMMENT 'Now - max_event_ts in minutes',
  null_rate          DOUBLE COMMENT 'Null fraction for key fields (computed)',
  duplicate_rate     DOUBLE COMMENT 'Duplicate fraction (computed)',
  status             STRING COMMENT 'OK/WARN/FAIL',
  computed_ts        TIMESTAMP COMMENT 'Metrics computed timestamp'
)
USING DELTA
PARTITIONED BY (metric_date)
COMMENT 'Gold observability: data freshness/volume/quality metrics';

**Quick sanity checks (optional)**

In [0]:
SHOW SCHEMAS IN enterprise;
SHOW TABLES IN enterprise.bronze_market;
SHOW TABLES IN enterprise.silver_market;
SHOW TABLES IN enterprise.gold_market;
SHOW TABLES IN enterprise.gold_observability;